# 01 – Data Source Layer

**Projekt:** Wohnungsmarkt-Radar Wien – Preis/Leistungs-Score für Mietwohnungen

**Layer:** Data Source Layer - Identifikation der Datenquellen & Methoden der Datengewinnung

**Beschreibung:** Dieses Notebook beschreibt, woher die Daten kommen und wie sie gewonnen werden. Es beschreibt die Webscraper, zeigt Beispiele der Rohdaten und gibt einen Überblick über Umfang und Qualität der Daten

**Fragestellung:** 
Ist eine Mietwohnung für den Bezirk preislich günstig, normal oder teuer? Dafür bilden wir einen Preis/Leistungsscore pro Wohnung aus Gesamtmiete, Fläche, Zimmeranzahl, Lage und die Anbindung an den öffentlichen Verkehr.

**Überblick der Quellen:**

| Quelle | Art | Methode | Format | Koordinaten |
|---|---|---|---|---|
| Willhaben | Web Scraping | HTML abrufen, eingebettetes JSON (`__NEXT_DATA__`) auslesen | JSONL | ja |
| ImmoScout24 | Web Scraping | GraphQL-Schnittstelle der Website (Persisted Query) | JSONL | ja |
| DerStandard Immobilien | Web Scraping | HTML abrufen, Karten und eingebettetes JSON auslesen | JSONL | nein (Geocoding nötig) |
| Wiener Linien: Haltestellen | Open Data | Download GeoJSON | GeoJSON | ja |
| Wien: Bezirksgrenzen | Open Data | Download GeoJSON | GeoJSON | ja (EPSG:31256) |
| Nominatim (OpenStreetMap) | Geocoding | Adresse -> Koordinaten, nur für DerStandard | JSON (Cache) | ja |

**Links zu den Quellen:**

•	ImmoScout24: https://www.immobilienscout24.at/portal/graphql

•	Der Standard: https://immobilien.derstandard.at/suche/wien/mieten-wohnung 

•	Willhaben: https://www.willhaben.at/iad/immobilien/mietwohnungen/mietwohnung-angebote?areaId=900

•	Öffentliches Verkehrsnetz Haltestellen Wien: https://www.data.gv.at/datasets/f1f6f15d-2faa-4b62-b78b-80599dd1c66e?locale=de

•	Bezirksgrenzen: https://www.data.gv.at/datasets/2ee6b8bf-6292-413c-bb8b-bd22dbb2ad4b?locale=de





**Gesamtablauf:**
```
 Portale (Willhaben, ImmoScout24, DerStandard)      Open Data (Haltestellen, Bezirke)
              │  Node.js-Scraper (app.js)                        │  Download aus data.gv.at
              ▼                                                  ▼
        data/raw/*.jsonl                                data/external/*.geojson
              │                                                  │
              └──────────────► 02_data_storage ◄─────────────────┘





# 1. Vorbereitung

Pfad definieren, Dateien einlesen und Daten ausgeben

In [3]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

BASE = Path.cwd().parent
RAW = BASE / "data" / "raw"
EXTERNAL = BASE / "data" / "external"


def read_jsonl(path):
    """Liest eine JSONL-Datei (eine JSON-Zeile pro Inserat)."""
    if not path.exists():
        print(f"{path} nicht gefunden – bitte zuerst 'node app.js' ausführen")
        return []
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


willhaben = read_jsonl(RAW / "willhaben_listings.jsonl")
immoscout = read_jsonl(RAW / "immobilienscout_listings.jsonl")
standard = read_jsonl(RAW / "standard_listings.jsonl")
print(f"Willhaben: {len(willhaben)} | ImmoScout24: {len(immoscout)} | DerStandard: {len(standard)} Inserate")

Willhaben: 125 | ImmoScout24: 100 | DerStandard: 76 Inserate


# 2. Webscraping der Portale

Die Scraper sind in Node.js geschrieben und liegen im Ordner immo_webscraper

app.js <- zentrale Steuerung der einzelnen Scraper mit node app.js
Die Scraper werden nacheinander ausgeführt:
test/willhaben_test.js      <- scrapeWillhaben()
test/immoscout24_test.js    <- scrapeImmoScout()
test/standard_test.js       <- scrapeStandard()

Jeder Scraper wird nacheinander ausgeführt und lässt 3 Sekunden Pause zwischen den Portalseiten, da sonst ein Timeout (zumindest auf Willhaben) kommt, wenn diese zu schnell gescraped werden. Die Daten werden unter data/raw als JSONL gespeichert und nicht verändert. 

**Willhaben Scraper:**
Der Scraper lädt die Suchseite, schneiden den __NEXT_DATA__ -Block aus, parst ihn und wandelt das Attribut-Array jedes Inserats (zB: {name: "PRICE", values: [...]}) in Felder wie price, estate_size ect. um. Auf Willhaben sind coordinates sowie floor vorhanden.

**ImmoScout24 Scraper:**
Die Seite lädt ihre Ergebnisse über eine GraphQL-Schnittstelle (`/portal/graphql`). Der Scraper schickt dieselbe Anfrage wie die Website (Parameter: Miete, Wohnnutzung, Region Wien, Seitengröße, Startindex) und speichert die Treffer. Koordinaten sind vorhanden (`location.lat/lon`), ein Stockwerk nicht konsistent. In der Stichprobe enthielt die Antwort teils Kauf-Inserate und Bauprojekte mit Spannen statt Einzelwerten. Diese werden erst in der Bereinigung aussortiert.

**Standard-Scraper:**
Die Suchseite liefert die Inserate als HTML-Karten und zusätzlich als eingebettetes JSON (`PropertyEntryResponse`). Der Scraper liest
- aus den Karten: Titel, Gesamtmiete, Fläche, Zimmer, Ort,
- aus dem eingebetteten JSON: Straße, PLZ, Objektart, Ausstattung (z. B. Balkon, Aufzug).

Einschränkung: DerStandard liefert keine Koordinaten. Etwa die Hälfte der Inserate hat außerdem keine Straße. Deshalb könnte die Position in der Bereinigung über Geocoding (Nominatim) oder, als grobe Näherung, über den PLZ-Mittelpunkt bestimmt werden oder aber auch komplett aus dem Scope entfernt werden.
